# 03 — Interpretabilidade do Modelo
**Tech Challenge Fase 3 — FIAP IA Scientist**

Este notebook explica **por que** o modelo toma suas decisões,
usando SHAP Values para identificar os fatores que mais influenciam
a probabilidade de um município estar em risco de não atingir a meta.

## Por que interpretabilidade importa?

Um modelo de 96% de accuracy que ninguém entende não serve para
política pública. Gestores precisam saber **quais fatores atacar**
para reduzir o risco de alfabetização em seus municípios.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
import joblib
import json
from pathlib import Path
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")

## 1. Carregamento do modelo salvo

In [ ]:
pipeline = joblib.load("models/modelo_final.joblib")

with open("models/metadata.json", encoding="utf-8") as f:
    metadata = json.load(f)

threshold = metadata["threshold"]
features_num = metadata["features_numericas"]
features_cat = metadata["features_categoricas"]

caminho = Path("data/processed/dataset_enriquecido_v2.parquet")
if not caminho.exists():
    caminho = Path("data/processed/dataset_modelagem_v2.parquet")
df = pd.read_parquet(caminho)

features_all = [f for f in features_num + features_cat if f in df.columns]
X = df[features_all]
y = df["em_risco_2024"]

_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Modelo: {metadata['modelo']}")
print(f"Conjunto de teste: {X_test.shape}")

## 2. Extração dos nomes reais das features

In [ ]:
preprocessador = pipeline.named_steps["pre"]
modelo = pipeline.named_steps["clf"]

X_test_proc = preprocessador.transform(X_test)

# Extrai nomes reais
nomes = list(features_num)
imputer = preprocessador.named_transformers_["num"].named_steps["imputer"]
if hasattr(imputer, "indicator_") and imputer.indicator_ is not None:
    for i in imputer.indicator_.features_:
        nomes.append(f"missing_{features_num[i]}")

if "cat" in preprocessador.named_transformers_:
    encoder = preprocessador.named_transformers_["cat"].named_steps["onehot"]
    nomes.extend(encoder.get_feature_names_out(features_cat))

if X_test_proc.shape[1] != len(nomes):
    nomes = [f"feature_{i}" for i in range(X_test_proc.shape[1])]

print(f"Features no modelo: {X_test_proc.shape[1]}")
print(f"Nomes extraídos: {len(nomes)}")

## 3. SHAP Values — Importância Global

In [ ]:
explainer = shap.TreeExplainer(modelo)
shap_values = explainer.shap_values(X_test_proc)

importancia = np.abs(shap_values).mean(axis=0)
indices = np.argsort(importancia)[::-1][:15]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(15), importancia[indices][::-1],
        color="steelblue", alpha=0.85)
ax.set_yticks(range(15))
ax.set_yticklabels([nomes[i] for i in indices[::-1]], fontsize=9)
ax.set_xlabel("Mean |SHAP Value|")
ax.set_title("Top 15 Features — Importância SHAP
Modelo v2 (Design Temporal Correto)")
plt.tight_layout()
plt.show()

print("Top 10 features:")
for i, idx in enumerate(indices[:10]):
    print(f"  {i+1:2d}. {nomes[idx]:<40} SHAP: {importancia[idx]:.4f}")

## 4. SHAP Summary Plot

O summary plot mostra não só a importância mas também o **sentido do impacto**:
vermelho = valor alto da feature, azul = valor baixo.
Pontos à direita aumentam o risco, à esquerda diminuem.

In [ ]:
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_proc,
                  feature_names=nomes,
                  max_display=15, show=False)
plt.title("SHAP Summary Plot — Impacto por Feature")
plt.tight_layout()
plt.show()

## 5. Análise de municípios individuais

O waterfall plot explica a predição para um município específico:
como cada feature empurra a probabilidade para cima ou para baixo.

In [ ]:
y_prob = pipeline.predict_proba(X_test)[:, 1]

# Município de alto risco
idx_alto = np.argmax(y_prob)
# Município de baixo risco
idx_baixo = np.argmin(y_prob)
# Município limítrofe
idx_limite = np.argmin(np.abs(y_prob - threshold))

for nome, idx in [("Alto risco", idx_alto),
                  ("Baixo risco", idx_baixo),
                  ("Limítrofe", idx_limite)]:
    prob = y_prob[idx]
    real = y_test.iloc[idx]
    print(f"{nome}: prob={prob:.3f} | real={real}")

## Conclusões da Interpretabilidade

- **media_pt_2023** é o preditor dominante — desempenho em português em 2023
  prediz fortemente o risco em 2024
- **Efeitos regionais** (sigla_uf_RS, sigla_uf_BA, regiao_Sudeste) capturam
  variabilidade não explicada por outras features
- **particip_2023** confirma H4 — participação como proxy de gestão municipal
- **taxa_alf_2023** confirma H1 — inércia histórica relevante

### Implicação para política pública
Municípios com  baixa E  baixa são os
candidatos de maior risco composto — intervenção em língua portuguesa
é a alavanca principal identificada pelo modelo.